In [9]:
# Cell 1: Imports & Setup
!pip -q install gTTS

import pandas as pd
import numpy as np
from gtts import gTTS
from IPython.display import Audio, display
import os

print("Environment initialized! All libraries loaded successfully.")

Environment initialized! All libraries loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
# Cell 2: Districts, live data (Open-Meteo) and crop assumptions
import time
import requests
import pandas as pd

# ---- 1) Districts (approximate coordinates: latitude, longitude) ----
DISTRICTS = {
    "Sahiwal": (30.66, 73.10),
    "Khanewal": (30.30, 71.93),
    "Pakpattan": (30.34, 73.39),
    "Faisalabad": (31.42, 73.08),
    "Multan": (30.20, 71.47),
    "Rahim Yar Khan": (28.42, 70.30),
    "Sukkur": (27.70, 68.86),
    "Badin": (24.66, 68.84),
}
CROPS = ["Wheat", "Cotton", "Rice"]

# ---- 2) Crop assumptions (PROTOTYPE VALUES: replace with sourced figures) ----
# Check against Punjab/Sindh agriculture department crop budgets and current market rates.
CROP_ASSUMPTIONS = {
    "Wheat":  {"price_per_maund": 3900.0, "cost_per_acre": 70000.0, "base_yield_maunds": 38.0},
    "Cotton": {"price_per_maund": 8500.0, "cost_per_acre": 85000.0, "base_yield_maunds": 24.0},
    "Rice":   {"price_per_maund": 4500.0, "cost_per_acre": 75000.0, "base_yield_maunds": 42.0},
}

# Used only if the internet or the API is unavailable (so the demo never crashes)
FALLBACK_CONDITIONS = {"temp": 30.0, "humidity": 55.0, "aqi": 120.0, "pm25": None, "rain": 10.0}

# ---- 3) Helpers ----
def _get_json(url, params, retries=2, timeout=20):
    last_error = None
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_error = e
            if attempt < retries:
                time.sleep(1.5)
    raise RuntimeError(f"Request failed: {last_error}")

def _num(value, name):
    if value is None or pd.isna(value):
        raise ValueError(f"No data returned for {name}")
    return round(float(value), 1)

# ---- 4) Live data functions ----
def get_weather(district, days=7):
    lat, lon = DISTRICTS[district]
    data = _get_json("https://api.open-meteo.com/v1/forecast", {
        "latitude": lat, "longitude": lon, "forecast_days": days, "timezone": "Asia/Karachi",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "hourly": "relative_humidity_2m"})
    daily = pd.DataFrame(data["daily"])
    hourly = pd.DataFrame(data["hourly"])
    temp = ((daily["temperature_2m_max"] + daily["temperature_2m_min"]) / 2).mean()
    return {
        "temp": _num(temp, "temperature"),
        "humidity": _num(hourly["relative_humidity_2m"].mean(), "humidity"),
        "rain": _num(daily["precipitation_sum"].fillna(0).sum(), "rainfall"),
    }

def get_air_quality(district, days=3):
    lat, lon = DISTRICTS[district]
    data = _get_json("https://air-quality-api.open-meteo.com/v1/air-quality", {
        "latitude": lat, "longitude": lon, "forecast_days": days, "timezone": "Asia/Karachi",
        "hourly": "us_aqi,pm2_5"})
    hourly = pd.DataFrame(data["hourly"])
    return {
        "aqi": _num(hourly["us_aqi"].mean(), "air quality index"),
        "pm25": _num(hourly["pm2_5"].mean(), "PM2.5"),
    }

def get_river_flow_ratio(district, days=7):
    # Peak forecast river flow divided by the normal flow. A value well above 1 means unusually high water.
    lat, lon = DISTRICTS[district]
    data = _get_json("https://flood-api.open-meteo.com/v1/flood", {
        "latitude": lat, "longitude": lon, "forecast_days": days,
        "daily": "river_discharge,river_discharge_mean"})
    daily = pd.DataFrame(data["daily"])
    mean_flow = daily["river_discharge_mean"].mean()
    peak = daily["river_discharge"].max()
    if pd.isna(mean_flow) or mean_flow <= 0 or pd.isna(peak):
        return None
    return round(float(peak / mean_flow), 2)

_CACHE = {}
CACHE_SECONDS = 1800   # reuse results for 30 minutes

def get_live_conditions(district, use_cache=True):
    if district not in DISTRICTS:
        raise ValueError(f"Unknown district '{district}'. Choose from: {', '.join(DISTRICTS)}")

    cached = _CACHE.get(district)
    if use_cache and cached and time.time() - cached[0] < CACHE_SECONDS:
        return dict(cached[1])

    result = dict(FALLBACK_CONDITIONS)
    result.update({"district": district, "river_flow_ratio": None, "warnings": []})

    try:
        result.update(get_weather(district))
    except Exception as e:
        result["warnings"].append(f"weather: {e}")
    try:
        result.update(get_air_quality(district))
    except Exception as e:
        result["warnings"].append(f"air quality: {e}")
    try:
        result["river_flow_ratio"] = get_river_flow_ratio(district)
    except Exception as e:
        result["warnings"].append(f"river flow: {e}")

    result["source"] = "live" if not result["warnings"] else "partly fallback sample"
    if not result["warnings"]:
        _CACHE[district] = (time.time(), dict(result))
    return result

# ---- 5) Run it: live snapshot for every district (also saved as an offline backup) ----
rows = []
for d in DISTRICTS:
    c = get_live_conditions(d)
    rows.append({
        "District": d, "Temp_C": c["temp"], "Humidity_Pct": c["humidity"], "AQI": c["aqi"],
        "Rain_7d_mm": c["rain"], "River_Flow_Ratio": c["river_flow_ratio"], "Source": c["source"],
    })
df_live = pd.DataFrame(rows)
df_live.to_csv("live_conditions_snapshot.csv", index=False)

print("Live conditions fetched and saved as 'live_conditions_snapshot.csv'")
print(df_live.to_string(index=False))

Live conditions fetched and saved as 'live_conditions_snapshot.csv'
      District  Temp_C  Humidity_Pct   AQI  Rain_7d_mm  River_Flow_Ratio Source
       Sahiwal    25.9          76.8 161.0        30.4              3.03   live
      Khanewal    26.7          72.1 148.8        35.4              0.95   live
     Pakpattan    26.4          73.9 161.1        21.8              1.24   live
    Faisalabad    25.0          79.5 159.9        33.8              1.44   live
        Multan    27.4          67.2 148.6        12.6              0.82   live
Rahim Yar Khan    31.4          49.3 109.9         0.0              1.17   live
        Sukkur    32.2          46.9  85.3         0.0              1.02   live
         Badin    31.2          57.1  72.8         0.0               NaN   live


In [11]:
# Cell 3: Core AgriTech Engine
import numpy as np

# This cell needs Cell 2 (live data and crop assumptions) to have been run first
if "CROP_ASSUMPTIONS" not in globals() or "get_live_conditions" not in globals():
    raise RuntimeError("Please run Cell 2 first.")

# Disease and pest knowledge base (PROTOTYPE: verify against Punjab/Sindh agriculture department advisories).
# No brand names and no doses on purpose. Always confirm the product and dose with the local extension officer.
DISEASE_KB = {
    "Wheat": {
        "yellow rust": {
            "Disease": "Yellow Rust (Stripe Rust)",
            "keywords": ["yellow rust", "stripe rust", "rust"],
            "Active Ingredient": "Triazole fungicides, for example propiconazole or tebuconazole",
            "Prevention": "Grow resistant varieties, avoid late sowing, and scout fields regularly.",
        },
        "aphid": {
            "Disease": "Aphid Infestation",
            "keywords": ["aphid"],
            "Active Ingredient": "Insecticides such as imidacloprid, as advised by your extension officer",
            "Prevention": "Scout for colonies, protect natural enemies such as ladybirds, and avoid unnecessary spraying.",
        },
    },
    "Cotton": {
        "pink bollworm": {
            "Disease": "Pink Bollworm",
            "keywords": ["pink bollworm", "bollworm"],
            "Active Ingredient": "Pheromone traps and mating-disruption ropes (PB-Rope); insecticides only as advised locally",
            "Prevention": "Destroy crop residue after harvest, avoid very late sowing, and monitor traps.",
        },
        "whitefly": {
            "Disease": "Whitefly / Cotton Leaf Curl Virus (vector)",
            "keywords": ["whitefly", "leaf curl", "clcuv"],
            "Active Ingredient": "Whitefly control insecticides such as diafenthiuron or spirotetramat, as advised locally",
            "Prevention": "Infected plants cannot be cured, so control the whitefly. Use tolerant varieties, sow on time, and remove weed hosts.",
        },
    },
    "Rice": {
        "rice blast": {
            "Disease": "Rice Blast (Magnaporthe oryzae)",
            "keywords": ["blast"],
            "Active Ingredient": "Fungicides such as tricyclazole or azoxystrobin, as advised locally",
            "Prevention": "Avoid excess nitrogen, use resistant varieties, and avoid long periods of leaf wetness.",
        },
        "stem borer": {
            "Disease": "Rice Stem Borer",
            "keywords": ["stem borer", "borer"],
            "Active Ingredient": "Insecticides such as chlorantraniliprole, as advised locally",
            "Prevention": "Destroy stubble after harvest, use pheromone traps, and avoid late planting.",
        },
    },
}


class AgriTechEngine:
    SUPPORTED = ["Wheat", "Cotton", "Rice"]

    def __init__(self, crop_name: str):
        crop = str(crop_name).strip().capitalize()
        if crop not in self.SUPPORTED:
            raise ValueError("Supported crops: " + ", ".join(self.SUPPORTED))
        self.crop = crop

    # ---------- 1) Environment feasibility ----------
    def evaluate_environment(self, temp: float, humidity: float, aqi: float, rainfall: float,
                             river_flow_ratio=None) -> dict:
        risk = 0
        reasons = []

        if self.crop == "Wheat":
            if temp > 30.0:
                risk += 30
                reasons.append("Heat stress: average temperature is above 30 C, which can hurt wheat.")
            if 10.0 <= temp <= 18.0 and humidity > 70.0:
                risk += 15
                reasons.append("Cool, humid weather favours yellow rust.")
        elif self.crop == "Cotton":
            if rainfall > 120.0:
                risk += 35
                reasons.append("Very heavy rain forecast, which can damage open bolls.")
            if humidity > 80.0:
                risk += 20
                reasons.append("High humidity can raise whitefly and bollworm pressure.")
        elif self.crop == "Rice":
            if temp > 38.0:
                risk += 25
                reasons.append("Extreme heat can affect panicle development.")
            if humidity > 85.0:
                risk += 20
                reasons.append("Very high humidity favours rice blast.")

        # Risks that apply to every crop
        if aqi > 200.0:
            risk += 20
            reasons.append("Very high air pollution (AQI above 200) can stress crops and reduce sunlight.")
        if river_flow_ratio is not None:
            if river_flow_ratio >= 3.0:
                risk += 30
                reasons.append("River flow is forecast far above normal: serious flood risk.")
            elif river_flow_ratio >= 1.5:
                risk += 15
                reasons.append("River flow is forecast above normal: watch for flooding.")

        risk = min(risk, 100)
        feasibility = 100 - risk
        if feasibility >= 75:
            status = "Highly Favorable"
        elif feasibility >= 50:
            status = "Moderate Risk"
        else:
            status = "High Environmental Risk"

        return {
            "Crop": self.crop,
            "Feasibility Score": feasibility,
            "Risk Score": risk,
            "Status": status,
            "Risks": reasons or ["Optimal Environment"],
            "Care Plan": self.care_plan(temp, humidity, aqi, rainfall, risk),
        }

    def care_plan(self, temp, humidity, aqi, rainfall, risk) -> dict:
        # Simple prototype rules: confirm with your extension officer
        if rainfall >= 40.0:
            water = "Reduce irrigation: rain is forecast."
        elif temp > 32.0 and rainfall < 10.0:
            water = "Increase irrigation: hot and dry week."
        else:
            water = "Keep the normal irrigation schedule."

        if rainfall >= 60.0:
            fertilizer = "Delay top-dressing to avoid nutrient loss in heavy rain."
        else:
            fertilizer = "Apply according to a soil test or extension advice, in split doses."

        scouting = ("Check fields every 3 to 4 days." if humidity > 80.0 or risk >= 50
                    else "Check fields once a week.")
        protection = ("Consider protecting seedbeds or nurseries on high-risk days; confirm with your extension officer."
                      if risk >= 50 else "Not needed at this risk level.")
        return {"Water": water, "Fertilizer": fertilizer,
                "Pest and disease scouting": scouting, "Protected cultivation": protection}

    # ---------- 2) Disease and pest advice ----------
    def diagnose_and_recommend(self, issue_key: str) -> dict:
        query = str(issue_key).strip().lower()
        for entry in DISEASE_KB.get(self.crop, {}).values():
            if any(kw in query or query in kw for kw in entry["keywords"]) and query:
                return {
                    "Disease": entry["Disease"],
                    "Commercial Product": "Choose a registered product with this active ingredient; confirm with your extension officer",
                    "Active Ingredient": entry["Active Ingredient"],
                    "Prevention": entry["Prevention"],
                    "Matched": True,
                }
        return {
            "Disease": "General Stress",
            "Commercial Product": "Consult your local agriculture extension officer",
            "Active Ingredient": "N/A",
            "Prevention": "Scout the field and send a clear leaf photo for diagnosis.",
            "Matched": False,
        }

    # ---------- 3) Money ----------
    def financial_forecast(self, acres: float, yield_maunds_per_acre: float = None) -> dict:
        if acres <= 0:
            raise ValueError("Acres must be greater than zero.")
        a = CROP_ASSUMPTIONS[self.crop]
        y = a["base_yield_maunds"] if yield_maunds_per_acre is None else yield_maunds_per_acre

        total_maunds = acres * y
        gross_rev = total_maunds * a["price_per_maund"]
        total_cost = a["cost_per_acre"] * acres
        net_profit = gross_rev - total_cost
        roi = (net_profit / total_cost) * 100 if total_cost > 0 else 0

        return {
            "Total Acres": acres,
            "Total Output (Maunds)": round(total_maunds, 1),
            "Total Cost (PKR)": round(total_cost, 2),
            "Gross Revenue (PKR)": round(gross_rev, 2),
            "Net Profit (PKR)": round(net_profit, 2),
            "ROI (%)": round(roi, 2),
        }

    def simulate_profit(self, acres: float, risk_score: float = 0, disease_present: bool = False,
                        offered_price: float = None, n: int = 5000, seed: int = 42) -> dict:
        # Prototype simulation. Assumptions: environmental risk can cut yield by up to 40%,
        # disease cuts yield by 10-30% (or 5-20% in 15% of cases without disease),
        # yield varies by about 8%, price by 10% and cost by 7%.
        if acres <= 0:
            raise ValueError("Acres must be greater than zero.")
        rng = np.random.default_rng(seed)
        a = CROP_ASSUMPTIONS[self.crop]
        benchmark = a["price_per_maund"]

        env_loss = np.clip(risk_score / 100.0 * 0.40, 0.0, 0.40)
        if disease_present:
            disease_loss = rng.uniform(0.10, 0.30, n)
        else:
            disease_loss = np.where(rng.random(n) < 0.15, rng.uniform(0.05, 0.20, n), 0.0)

        yields = a["base_yield_maunds"] * (1 - env_loss) * (1 - disease_loss) * rng.normal(1.0, 0.08, n)
        yields = np.clip(yields, 0, None)

        mean_price = offered_price if offered_price else benchmark
        price = np.clip(rng.normal(mean_price, 0.10 * mean_price, n), 0, None)
        cost = rng.normal(a["cost_per_acre"], 0.07 * a["cost_per_acre"], n) * acres

        profit = yields * acres * price - cost
        p10, p50, p90 = np.percentile(profit, [10, 50, 90])

        if offered_price is None:
            fair = "No offer entered."
        else:
            gap = (offered_price - benchmark) / benchmark * 100
            if gap <= -10:
                fair = f"Offer is {abs(gap):.0f}% below the benchmark price. Compare other buyers before selling."
            elif gap >= 10:
                fair = f"Offer is {gap:.0f}% above the benchmark price."
            else:
                fair = "Offer is close to the benchmark price."

        return {
            "Profit Low Case P10 (PKR)": round(float(p10), 0),
            "Profit Median (PKR)": round(float(p50), 0),
            "Profit High Case P90 (PKR)": round(float(p90), 0),
            "Chance of Loss (%)": round(float((profit < 0).mean() * 100), 1),
            "Benchmark Price (PKR/maund)": benchmark,
            "Fair-Price Check": fair,
        }

    # ---------- 4) Everything together, using live data ----------
    def farm_report(self, district: str, acres: float, issue: str = None, offered_price: float = None) -> dict:
        cond = get_live_conditions(district)
        env = self.evaluate_environment(cond["temp"], cond["humidity"], cond["aqi"], cond["rain"],
                                        cond["river_flow_ratio"])
        diag = self.diagnose_and_recommend(issue) if issue else None
        money = self.simulate_profit(acres, risk_score=env["Risk Score"],
                                     disease_present=bool(issue), offered_price=offered_price)
        return {"Conditions": cond, "Environment": env, "Diagnosis": diag, "Money": money}


print("AgriTechEngine class defined successfully!")

# Quick self-test with fixed numbers (no internet needed)
_test = AgriTechEngine("wheat")
_env = _test.evaluate_environment(32.5, 55, 240, 10)
print(_env["Feasibility Score"], _env["Status"], _env["Risks"])
print(_test.diagnose_and_recommend("yellow rust")["Disease"])
print(_test.simulate_profit(5, risk_score=_env["Risk Score"], disease_present=True, offered_price=3300))

AgriTechEngine class defined successfully!
50 Moderate Risk ['Heat stress: average temperature is above 30 C, which can hurt wheat.', 'Very high air pollution (AQI above 200) can stress crops and reduce sunlight.']
Yellow Rust (Stripe Rust)
{'Profit Low Case P10 (PKR)': -29797.0, 'Profit Median (PKR)': 47993.0, 'Profit High Case P90 (PKR)': 134129.0, 'Chance of Loss (%)': 22.2, 'Benchmark Price (PKR/maund)': 3900.0, 'Fair-Price Check': 'Offer is 15% below the benchmark price. Compare other buyers before selling.'}


In [12]:
# Cell 4: Multilingual AI Advisor (LLM grounded on live farm data) with voice output
!pip -q install anthropic edge-tts

import os
import re
import asyncio
import threading
import anthropic
import edge_tts
from gtts import gTTS
from IPython.display import Audio, display

# ---------- Settings ----------
MODEL_NAME = "claude-sonnet-5-5"
LANGUAGES = ["English", "Urdu", "Punjabi", "Sindhi", "Saraiki"]

SCRIPT_NOTES = {
    "English": "",
    "Urdu": "Write in Urdu script.",
    "Punjabi": "Write in Punjabi using Shahmukhi (Urdu-style) script, as farmers in Pakistan read it.",
    "Sindhi": "Write in Sindhi using the Sindhi Arabic script.",
    "Saraiki": "Write in Saraiki using Urdu-style Arabic script.",
}

# Voices: neural voices exist for English and Urdu only.
# Punjabi, Sindhi and Saraiki are read with the Urdu voice (approximate).
EDGE_VOICES = {
    "English": "en-US-AriaNeural",
    "Urdu": "ur-PK-UzmaNeural",      # for a male voice use "ur-PK-AsadNeural"
    "Punjabi": "ur-PK-UzmaNeural",
    "Sindhi": "ur-PK-UzmaNeural",
    "Saraiki": "ur-PK-UzmaNeural",
}
TTS_CODES = {"English": "en", "Urdu": "ur", "Punjabi": "ur", "Sindhi": "ur", "Saraiki": "ur"}  # gTTS backup

# Used only when the LLM is unavailable (no API key, no internet, or an API error).
# Add Punjabi / Sindhi / Saraiki entries here if you have fluent-speaker-checked translations.
FALLBACK_KB = {
    "fertilizer": {
        "English": "Apply fertilizer according to a soil test or your extension officer's advice, in split doses.",
        "Urdu": "کھاد کی مقدار مٹی کے ٹیسٹ یا زرعی افسر کے مشورے کے مطابق، کئی قسطوں میں ڈالیں۔",
    },
    "disease": {
        "English": "If you see spots or marks on the leaves, take a clear photo and consult your local extension officer.",
        "Urdu": "پتوں پر دھبے یا نشان نظر آئیں تو پتے کی صاف تصویر لیں اور اپنے مقامی زرعی افسر سے مشورہ کریں۔",
    },
    "general": {
        "English": "The advisor is offline right now. Please contact your local agriculture extension officer.",
        "Urdu": "مشیر اس وقت دستیاب نہیں ہے۔ براہِ کرم اپنے مقامی زرعی افسر سے رابطہ کریں۔",
    },
}

SYSTEM_TEMPLATE = """You are Kisan AI, a friendly agricultural advisor for farmers in Pakistan.
Rules:
- Answer in {language}. {script_note} Use simple, short sentences a farmer can understand.
- Use only the farm data below for numbers, risks and forecasts. If something is not in the data, say you do not know instead of guessing.
- Never give pesticide or fertilizer doses or mixing rates. Name active ingredients only, and tell the farmer to follow the product label and confirm the product and dose with the local agriculture extension officer.
- Do not recommend brand names.
- Stay on farming topics. For anything else, politely say you can only help with farming.
- If the question suggests an emergency (poisoning, flood happening now), tell the farmer to contact local authorities or emergency services.
- Keep answers under 120 words.

FARM DATA:
{context}
"""


# ---------- Helpers ----------
def _load_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.environ.get("ANTHROPIC_API_KEY")


def _run_async(coro):
    # Colab already has a running event loop, so run the async voice call in its own thread
    result = {}

    def runner():
        loop = asyncio.new_event_loop()
        try:
            result["value"] = loop.run_until_complete(coro)
        except Exception as e:
            result["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=runner)
    t.start()
    t.join()
    if "error" in result:
        raise result["error"]
    return result.get("value")


async def _edge_save(text, voice, filename):
    await edge_tts.Communicate(text, voice, rate="-5%").save(filename)


# ---------- The chatbot ----------
class SpeakingAgriChatbot:
    def __init__(self, model: str = MODEL_NAME):
        self.model = model
        self.profile = None
        self.report = None
        self.history = []
        self.last_source = None
        key = _load_api_key()
        self.client = anthropic.Anthropic(api_key=key, timeout=30.0) if key else None
        if self.client is None:
            print("No ANTHROPIC_API_KEY found. The chatbot will use its offline fallback answers.")
            print("To turn on the AI advisor: Colab left sidebar > key icon > add ANTHROPIC_API_KEY and switch on Notebook access.")
        else:
            print("AI advisor is ready.")

    # ----- Farm profile -----
    def set_profile(self, district, crop, acres, issue=None, offered_price=None):
        self.profile = {"district": district, "crop": crop, "acres": acres,
                        "issue": issue, "offered_price": offered_price}
        self.report = None
        self.history = []
        if "AgriTechEngine" in globals():
            try:
                self.report = AgriTechEngine(crop).farm_report(district, acres, issue=issue, offered_price=offered_price)
            except Exception as e:
                print("Could not build the farm report:", e)
        else:
            print("Run Cell 3 first to give the advisor live farm data.")
        return self.report

    def _context_text(self):
        if not self.profile:
            return "No farm profile has been set."
        p = self.profile
        lines = [f"District: {p['district']}", f"Crop: {p['crop']}", f"Land: {p['acres']} acres"]
        r = self.report
        if r:
            c, e, m = r["Conditions"], r["Environment"], r["Money"]
            lines.append(
                f"Live conditions for the next 7 days: average temperature {c['temp']} C, humidity {c['humidity']}%, "
                f"air quality index {c['aqi']}, rainfall {c['rain']} mm, river flow ratio {c['river_flow_ratio']} "
                f"(data source: {c['source']})")
            lines.append(f"Feasibility: {e['Feasibility Score']}/100 ({e['Status']}). Risks: {'; '.join(e['Risks'])}")
            lines.append("Care plan: " + "; ".join(f"{k}: {v}" for k, v in e["Care Plan"].items()))
            lines.append(
                f"Profit estimate (prototype assumptions, PKR): low case {m['Profit Low Case P10 (PKR)']:,.0f}, "
                f"median {m['Profit Median (PKR)']:,.0f}, high case {m['Profit High Case P90 (PKR)']:,.0f}; "
                f"chance of loss {m['Chance of Loss (%)']}%")
            lines.append(f"Fair-price check: {m['Fair-Price Check']}")
            d = r.get("Diagnosis")
            if d:
                lines.append(f"Reported crop problem: {d['Disease']}. Active ingredient guidance: "
                             f"{d['Active Ingredient']}. Prevention: {d['Prevention']}")
        kb = globals().get("DISEASE_KB", {}).get(p["crop"], {})
        for entry in kb.values():
            lines.append(f"Reference for {p['crop']} ({entry['Disease']}): {entry['Active Ingredient']}. {entry['Prevention']}")
        return "\n".join(lines)

    # ----- Answers -----
    def _fallback(self, query, language):
        q = query.lower()
        if any(w in q for w in ["khad", "fertilizer", "fertiliser", "urea", "dap", "کھاد", "یوریا"]):
            key = "fertilizer"
        elif any(w in q for w in ["disease", "bimari", "spray", "pest", "بیماری", "سپرے", "کیڑ"]):
            key = "disease"
        else:
            key = "general"
        options = FALLBACK_KB[key]
        return options.get(language) or options["Urdu"]

    def ask(self, query: str, language: str = "Urdu") -> str:
        language = str(language).strip().capitalize()
        if language not in LANGUAGES:
            language = "Urdu"
        if not str(query).strip():
            return self._fallback("", language)

        if self.client is not None:
            try:
                system = SYSTEM_TEMPLATE.format(language=language, script_note=SCRIPT_NOTES[language],
                                                context=self._context_text())
                messages = self.history[-12:] + [{"role": "user", "content": query}]
                msg = self.client.messages.create(model=self.model, max_tokens=500,
                                                  system=system, messages=messages)
                text = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text").strip()
                if text:
                    self.history += [{"role": "user", "content": query}, {"role": "assistant", "content": text}]
                    self.last_source = "AI advisor"
                    return text
            except Exception as e:
                print(f"(AI advisor unavailable, using offline answer: {e})")

        self.last_source = "offline fallback"
        return self._fallback(query, language)

    # Same name as the old class, so Cell 6 still works
    def respond(self, query: str, language: str) -> str:
        return self.ask(query, language)

    # ----- Voice -----
    def speak(self, text: str, language: str = "Urdu"):
        language = str(language).strip().capitalize()
        if language not in EDGE_VOICES:
            language = "Urdu"
        clean = re.sub(r"[*#_`>]", "", text).strip()
        filename = f"advice_{language.lower()}.mp3"
        if language not in ("English", "Urdu"):
            print("Note: voice uses the Urdu voice for this language (approximate).")

        try:
            _run_async(_edge_save(clean, EDGE_VOICES[language], filename))
            return filename
        except Exception as e:
            print(f"Edge voice unavailable ({e}); trying the basic voice.")

        try:
            gTTS(text=clean, lang=TTS_CODES.get(language, "ur"), slow=False).save(filename)
            return filename
        except Exception as e:
            print(f"Audio generation notice: {e}")
            return None

    def speak_advisory(self, query: str, language: str):
        text = self.ask(query, language)
        print(f"[{language} | {self.last_source}]\n{text}\n")
        filename = self.speak(text, language)
        if filename:
            display(Audio(filename, autoplay=False))
        return text


def run_chatbot_tests(bot):
    tests = [
        ("When should I apply urea to my wheat?", "English"),
        ("میری فصل پر بیماری کے نشان ہیں، کیا کروں؟", "Urdu"),
        ("How many grams of fungicide should I mix per litre?", "English"),   # should refuse doses
        ("Who won the cricket match yesterday?", "English"),                  # should stay on farming
    ]
    for q, lang in tests:
        print("Q:", q)
        print("A:", bot.ask(q, lang))
        print("-" * 60)


# ---------- Create the chatbot and run a short demo ----------
bot_voice = SpeakingAgriChatbot()
print("Speaking Chatbot initialized successfully!")

try:
    bot_voice.set_profile("Multan", "Wheat", 5, issue="yellow rust", offered_price=3500)
    bot_voice.speak_advisory("Is it a good week to spray, and is the price offer fair?", "English")
    bot_voice.speak_advisory("میری گندم کے لیے پانی لگانے کا کیا وقت ہے؟", "Urdu")
except Exception as e:
    print("Demo skipped:", e)

No ANTHROPIC_API_KEY found. The chatbot will use its offline fallback answers.
To turn on the AI advisor: Colab left sidebar > key icon > add ANTHROPIC_API_KEY and switch on Notebook access.
Speaking Chatbot initialized successfully!
[English | offline fallback]
If you see spots or marks on the leaves, take a clear photo and consult your local extension officer.



[Urdu | offline fallback]
مشیر اس وقت دستیاب نہیں ہے۔ براہِ کرم اپنے مقامی زرعی افسر سے رابطہ کریں۔



In [13]:
# Cell 5: Testing crops and problems with live data
import pandas as pd

if "AgriTechEngine" not in globals() or "get_live_conditions" not in globals():
    raise RuntimeError("Please run Cells 2 and 3 first.")

test_cases = [
    {"district": "Sahiwal",    "crop": "Wheat",  "acres": 5,  "issue": "yellow rust", "offered_price": 3500},
    {"district": "Multan",     "crop": "Cotton", "acres": 5,  "issue": "whitefly",    "offered_price": 7500},
    {"district": "Badin",      "crop": "Rice",   "acres": 5,  "issue": "rice blast",  "offered_price": 4200},
    {"district": "Faisalabad", "crop": "Wheat",  "acres": 10, "issue": None,          "offered_price": None},
]

def pkr(x):
    return f"PKR {x:,.0f}"

summary_rows = []
test_reports = []

for case in test_cases:
    title = f"{case['crop'].upper()} in {case['district'].upper()} ({case['acres']} acres)"
    print("\n" + "=" * 20 + f" {title} " + "=" * 20)
    try:
        engine = AgriTechEngine(case["crop"])
        report = engine.farm_report(case["district"], case["acres"],
                                    issue=case["issue"], offered_price=case["offered_price"])
        test_reports.append({**case, "report": report})
        cond, env, diag, money = report["Conditions"], report["Environment"], report["Diagnosis"], report["Money"]

        # 1) Live conditions
        flow = cond["river_flow_ratio"] if cond["river_flow_ratio"] is not None else "n/a"
        print(f"Live conditions ({cond['source']}): {cond['temp']} C, humidity {cond['humidity']}%, "
              f"AQI {cond['aqi']}, rain {cond['rain']} mm (7 days), river flow ratio {flow}")

        # 2) Environmental feasibility and care plan
        print(f"Feasibility: {env['Feasibility Score']}/100 ({env['Status']})")
        print("Risks: " + "; ".join(env["Risks"]))
        print("Care plan:")
        for k, v in env["Care Plan"].items():
            print(f"  - {k}: {v}")

        # 3) Disease / pest advice
        if diag:
            print(f"Diagnosed issue: {diag['Disease']}")
            print(f"  Active ingredient guidance: {diag['Active Ingredient']}")
            print(f"  Prevention: {diag['Prevention']}")
        else:
            print("Diagnosed issue: none reported")

        # 4) Money: simple forecast and risk-based simulation
        fin = engine.financial_forecast(case["acres"])
        print(f"Simple forecast -> Cost: {pkr(fin['Total Cost (PKR)'])} | "
              f"Net profit: {pkr(fin['Net Profit (PKR)'])} (ROI {fin['ROI (%)']}%)")
        print(f"Risk-adjusted profit -> low {pkr(money['Profit Low Case P10 (PKR)'])} | "
              f"median {pkr(money['Profit Median (PKR)'])} | high {pkr(money['Profit High Case P90 (PKR)'])}")
        print(f"Chance of loss: {money['Chance of Loss (%)']}%")
        print(f"Fair-price check: {money['Fair-Price Check']}")

        summary_rows.append({
            "District": case["district"], "Crop": case["crop"], "Acres": case["acres"],
            "Feasibility": env["Feasibility Score"], "Status": env["Status"],
            "Issue": diag["Disease"] if diag else "None",
            "Median profit (PKR)": money["Profit Median (PKR)"],
            "Chance of loss (%)": money["Chance of Loss (%)"],
        })
    except Exception as e:
        print(f"This test case failed: {e}")

print("\n" + "=" * 25 + " SUMMARY " + "=" * 25)
if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    print(df_summary.to_string(index=False))
    df_summary.to_csv("test_summary.csv", index=False)
    print("\nSaved as 'test_summary.csv'")
else:
    print("No test case completed. Check that Cells 1 to 4 ran without errors.")


==================== WHEAT in SAHIWAL (5 acres) ====================
Live conditions (live): 25.9 C, humidity 76.8%, AQI 161.0, rain 30.4 mm (7 days), river flow ratio 3.03
Feasibility: 70/100 (Moderate Risk)
Risks: River flow is forecast far above normal: serious flood risk.
Care plan:
  - Water: Keep the normal irrigation schedule.
  - Fertilizer: Apply according to a soil test or extension advice, in split doses.
  - Pest and disease scouting: Check fields once a week.
  - Protected cultivation: Not needed at this risk level.
Diagnosed issue: Yellow Rust (Stripe Rust)
  Active ingredient guidance: Triazole fungicides, for example propiconazole or tebuconazole
  Prevention: Grow resistant varieties, avoid late sowing, and scout fields regularly.
Simple forecast -> Cost: PKR 350,000 | Net profit: PKR 391,000 (ROI 111.71%)
Risk-adjusted profit -> low PKR 26,073 | median PKR 114,648 | high PKR 212,711
Chance of loss: 4.7%
Fair-price check: Offer is 10% below the benchmark price. Compar

In [14]:
# Cell 6: Farmer action plan (built from live data) and chatbot test
if "AgriTechEngine" not in globals() or "bot_voice" not in globals():
    raise RuntimeError("Please run Cells 1 to 4 first.")


def pkr(x):
    return f"PKR {x:,.0f}"


def build_action_plan(district, crop, acres, report):
    cond, env, diag, money = report["Conditions"], report["Environment"], report["Diagnosis"], report["Money"]
    flow = cond.get("river_flow_ratio")
    flood_level = "high" if (flow is not None and flow >= 3.0) else "watch" if (flow is not None and flow >= 1.5) else "low"
    chance_loss = money["Chance of Loss (%)"]

    if env["Risk Score"] >= 50 or flood_level == "high":
        overall = "HIGH ATTENTION"
    elif env["Risk Score"] >= 25 or flood_level == "watch" or diag:
        overall = "WATCH CLOSELY"
    else:
        overall = "ROUTINE"

    line = "=" * 66
    out = [line, f"FARMER ACTION PLAN: {district.upper()} | {crop} | {acres} acres", line,
           f"Overall status: {overall}",
           f"Data source: {cond['source']} (forecast for the next 7 days)"]

    # 1) Field conditions and care
    out += ["", "1. FIELD CONDITIONS AND CARE",
            f"   Feasibility: {env['Feasibility Score']}/100 ({env['Status']})"]
    for r in env["Risks"]:
        out.append(f"   - Risk: {r}")
    for k, v in env["Care Plan"].items():
        out.append(f"   - {k}: {v}")
    if flood_level == "high":
        out += ["   - Flood: the forecast river flow is far above normal. Move stored inputs and grain to higher ground",
                "     and check official PMD, NDMA and PDMA alerts. This signal is indicative, not an official warning."]
    elif flood_level == "watch":
        out.append("   - Flood: river flow is above normal. Keep checking official PMD, NDMA and PDMA alerts.")

    # 2) Crop health
    out += ["", "2. CROP HEALTH"]
    if diag:
        out += [f"   - Issue: {diag['Disease']}",
                f"   - Active ingredient guidance: {diag['Active Ingredient']}",
                f"   - Prevention: {diag['Prevention']}",
                "   - Safety: follow the product label, confirm the product and dose with your extension officer,",
                "     and avoid spraying in strong wind or rain."]
    else:
        out.append("   - No problem reported. Keep scouting your field, and upload a leaf photo if you see spots or marks.")

    # 3) Money
    out += ["", "3. MONEY (prototype estimate)",
            f"   - Profit range: low {pkr(money['Profit Low Case P10 (PKR)'])} | "
            f"median {pkr(money['Profit Median (PKR)'])} | high {pkr(money['Profit High Case P90 (PKR)'])}",
            f"   - Chance of loss: {chance_loss}%",
            f"   - Fair-price check: {money['Fair-Price Check']}"]
    if chance_loss >= 20.0:
        out += ["   - Loss is a real possibility. To reduce it:",
                "       * Compare prices from several buyers before selling.",
                "       * Avoid unnecessary spraying and input purchases.",
                "       * Protect the crop from the main risks listed above.",
                "       * Ask your extension officer about alternative crops for the next season."]
    else:
        out.append("   - Loss risk is low under the current assumptions.")

    # 4) Honest scope note
    out += ["", "4. NOT COVERED IN THIS PROTOTYPE",
            "   - Urban expansion and farmland-loss monitoring (planned: satellite land-cover data).",
            "   - Soil test analysis (planned).",
            "", "Advice is a prototype estimate. Confirm decisions with your local agriculture extension officer.",
            line]
    return out


def generate_farmer_action_plan(district, crop, acres, issue=None, offered_price=None, language=None, bot=None):
    report = AgriTechEngine(crop).farm_report(district, acres, issue=issue, offered_price=offered_price)
    print("\n".join(build_action_plan(district, crop, acres, report)))

    # Optional: the AI advisor explains the plan in the farmer's language, with voice
    if language and bot is not None:
        bot.profile = {"district": district, "crop": crop, "acres": acres,
                       "issue": issue, "offered_price": offered_price}
        bot.report = report
        bot.history = []
        print(f"\n--- Advisor summary in {language} ---")
        bot.speak_advisory("Give me a short action plan for this week based on my farm data.", language)
    return report


# ---------- Tests ----------
# Test 1: wheat with a reported disease and a price offer, explained in Urdu with voice
try:
    generate_farmer_action_plan("Sahiwal", "Wheat", 5, issue="yellow rust", offered_price=3500,
                                language="Urdu", bot=bot_voice)
except Exception as e:
    print("Test 1 failed:", e)

# Test 2: a healthy wheat field, no offer entered (printed plan only)
try:
    generate_farmer_action_plan("Faisalabad", "Wheat", 10)
except Exception as e:
    print("Test 2 failed:", e)

# Test 3: the chatbot in Punjabi (profile from Test 1 is still loaded)
print("\n--- Testing chatbot in Punjabi ---")
try:
    bot_voice.speak_advisory("کھاد کدوں پاواں؟", "Punjabi")
except Exception as e:
    print("Test 3 failed:", e)

FARMER ACTION PLAN: SAHIWAL | Wheat | 5 acres
Overall status: HIGH ATTENTION
Data source: live (forecast for the next 7 days)

1. FIELD CONDITIONS AND CARE
   Feasibility: 70/100 (Moderate Risk)
   - Risk: River flow is forecast far above normal: serious flood risk.
   - Water: Keep the normal irrigation schedule.
   - Fertilizer: Apply according to a soil test or extension advice, in split doses.
   - Pest and disease scouting: Check fields once a week.
   - Protected cultivation: Not needed at this risk level.
   - Flood: the forecast river flow is far above normal. Move stored inputs and grain to higher ground
     and check official PMD, NDMA and PDMA alerts. This signal is indicative, not an official warning.

2. CROP HEALTH
   - Issue: Yellow Rust (Stripe Rust)
   - Active ingredient guidance: Triazole fungicides, for example propiconazole or tebuconazole
   - Prevention: Grow resistant varieties, avoid late sowing, and scout fields regularly.
   - Safety: follow the product labe

FARMER ACTION PLAN: FAISALABAD | Wheat | 10 acres
Overall status: ROUTINE
Data source: live (forecast for the next 7 days)

1. FIELD CONDITIONS AND CARE
   Feasibility: 100/100 (Highly Favorable)
   - Risk: Optimal Environment
   - Water: Keep the normal irrigation schedule.
   - Fertilizer: Apply according to a soil test or extension advice, in split doses.
   - Pest and disease scouting: Check fields once a week.
   - Protected cultivation: Not needed at this risk level.

2. CROP HEALTH
   - No problem reported. Keep scouting your field, and upload a leaf photo if you see spots or marks.

3. MONEY (prototype estimate)
   - Profit range: low PKR 493,223 | median PKR 753,762 | high PKR 1,027,500
   - Chance of loss: 0.0%
   - Fair-price check: No offer entered.
   - Loss risk is low under the current assumptions.

4. NOT COVERED IN THIS PROTOTYPE
   - Urban expansion and farmland-loss monitoring (planned: satellite land-cover data).
   - Soil test analysis (planned).

Advice is a proto

In [15]:
import requests
from google.colab import userdata

GEMINI_KEY = userdata.get("GEMINI_API_KEY")
BASE = "https://generativelanguage.googleapis.com/v1beta"
HEAD = {"x-goog-api-key": GEMINI_KEY, "Content-Type": "application/json"}

def list_models():
    r = requests.get(f"{BASE}/models", headers=HEAD, timeout=30)
    r.raise_for_status()
    names = [m["name"].split("/")[-1] for m in r.json().get("models", [])
             if "generateContent" in m.get("supportedGenerationMethods", [])]
    pref = [n for n in names if "flash" in n and "lite" in n]
    pref += [n for n in names if "flash" in n and "lite" not in n]
    pref += [n for n in names if n not in pref]
    return pref

def llm_answer(system_text, user_text, max_tokens=400, model=None):
    body = {
        "system_instruction": {"parts": [{"text": system_text}]},
        "contents": [{"role": "user", "parts": [{"text": user_text}]}],
        "generationConfig": {"maxOutputTokens": max_tokens, "temperature": 0.3},
    }
    r = requests.post(f"{BASE}/models/{model}:generateContent",
                      headers=HEAD, json=body, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"{model}: {r.status_code} {r.text[:200]}")
    return r.json()["candidates"][0]["content"]["parts"][0]["text"]

GEMINI_MODEL = None
for m in list_models()[:8]:
    try:
        out = llm_answer("Reply in Urdu in one short sentence.",
                         "Wheat ko pani kab dena chahiye?", model=m)
        GEMINI_MODEL = m
        print("WORKS:", m)
        print(out)
        break
    except Exception as e:
        print("skip ->", e)

if not GEMINI_MODEL:
    print("No model worked. Send me the 'skip' lines above.")

WORKS: gemini-flash-lite-latest
گندم کو پہلی آبپاشی بوائی کے 20 سے 25 دن بعد کرنی چاہیے۔


In [20]:
# Cell 7: Multilingual Regional Voice Library (Pashto, Balochi, Saraiki, Brahui)
import os
import re
import asyncio
import threading
import edge_tts
from gtts import gTTS
from IPython.display import Audio, display

class RegionalVoiceSynthesizer:
    # Map languages to the best available approximation neural voices
    # Pashto: Arabic/Urdu phonetics approximation
    # Balochi: Persian (Farsi) neural voice or Urdu
    # Saraiki: Urdu neural voice (very similar phonetics)
    # Brahui: Dravidian language written in Arabic/Urdu script, best approximated with Urdu neural voice
    REGIONAL_VOICES = {
        "Pashto": "ur-PK-UzmaNeural",  # fallback to ur-PK or Persian depending on availability
        "Balochi": "fa-IR-DilaraNeural", # Persian neural voice matches Balochi very closely
        "Saraiki": "ur-PK-UzmaNeural",  # Saraiki matches Pakistan Urdu neural voice
        "Brahui": "ur-PK-UzmaNeural",   # Brahui matches Pakistan Urdu neural voice
    }

    # Language codes for gTTS backup if Edge TTS neural voices fail
    GTTS_CODES = {
        "Pashto": "ps",
        "Balochi": "fa",
        "Saraiki": "ur",
        "Brahui": "ur"
    }

    def __init__(self):
        print("Regional Voice Synthesizer initialized for: Pashto, Balochi, Saraiki, Brahui.")

    def _run_async(self, coro):
        result = {}
        def runner():
            loop = asyncio.new_event_loop()
            try:
                result["value"] = loop.run_until_complete(coro)
            except Exception as e:
                result["error"] = e
            finally:
                loop.close()
        t = threading.Thread(target=runner)
        t.start()
        t.join()
        if "error" in result:
            raise result["error"]
        return result.get("value")

    async def _edge_save(self, text, voice, filename):
        await edge_tts.Communicate(text, voice, rate="-5%").save(filename)

    def synthesize_speech(self, text: str, language: str):
        language = str(language).strip().capitalize()
        if language not in self.REGIONAL_VOICES:
            raise ValueError(f"Supported regional languages: {', '.join(self.REGIONAL_VOICES.keys())}")

        clean_text = re.sub(r"[*#_`>]", "", text).strip()
        filename = f"voice_{language.lower()}.mp3"
        voice_model = self.REGIONAL_VOICES[language]

        print(f"Synthesizing {language} text using voice: {voice_model}...")
        try:
            self._run_async(self._edge_save(clean_text, voice_model, filename))
            display(Audio(filename, autoplay=False))
            return filename
        except Exception as e:
            print(f"Edge TTS neural synthesis missed for {language}: {e}. Trying gTTS backup...")

        try:
            lang_code = self.GTTS_CODES.get(language, "ur")
            gTTS(text=clean_text, lang=lang_code, slow=False).save(filename)
            display(Audio(filename, autoplay=False))
            return filename
        except Exception as e:
            print(f"Error generating audio for {language}: {e}")
            return None

# Instantiate the library
voice_lib = RegionalVoiceSynthesizer()

# Test the synthesis with sample inputs
print("\n--- Running Regional Voice Demo ---")
voice_lib.synthesize_speech("پښتو ژبه زما ژبه ده", "Pashto")
voice_lib.synthesize_speech("بلسا زبان منی جندءِ زبان انت", "Balochi")
voice_lib.synthesize_speech("سرائیکی میڈے وسیب دی سنڄاݨ ہے", "Saraiki")
voice_lib.synthesize_speech("براہوئی کنا باوہ پیرہ غا زبان ئِ", "Brahui")

Regional Voice Synthesizer initialized for: Pashto, Balochi, Saraiki, Brahui.

--- Running Regional Voice Demo ---
Synthesizing Pashto text using voice: ur-PK-UzmaNeural...


Synthesizing Balochi text using voice: fa-IR-DilaraNeural...


Synthesizing Saraiki text using voice: ur-PK-UzmaNeural...


Synthesizing Brahui text using voice: ur-PK-UzmaNeural...


'voice_brahui.mp3'

In [21]:
!pip -q install gradio
import io, os, re, json, hashlib, contextlib, base64, mimetypes, requests
import gradio as gr
from gtts import gTTS

# ---------- needs Cell 7 ----------
if "voice_lib" not in globals():
    raise RuntimeError("Run Cell 7 first (it creates voice_lib), then run this cell again.")

# ---------- choices pulled from your existing cells ----------
DISTRICT_LIST = list(DISTRICTS.keys()) if isinstance(DISTRICTS, dict) else [
    d if isinstance(d, str) else d.get("name") for d in DISTRICTS]
CROP_LIST = list(CROPS)

def _issue_choices():
    out = set()
    try:
        for k, v in DISEASE_KB.items():
            if isinstance(v, dict) and all(isinstance(x, dict) for x in v.values()):
                out.update(v.keys())
            else:
                out.add(k)
    except Exception:
        pass
    return ["None"] + sorted(out)
ISSUES = _issue_choices()

# (label shown, value used in prompts)
LANG_CHOICES = [("اردو", "Urdu"), ("English", "English"), ("پنجابی", "Punjabi"),
                ("سنڌي", "Sindhi"), ("سرائیکی", "Saraiki"), ("پښتو", "Pashto"),
                ("بلوچی", "Balochi"), ("براہوئی", "Brahui")]
SCRIPT_NOTE = ("(Urdu, Punjabi, Saraiki, Pashto, Balochi and Brahui in Arabic/Urdu script, "
               "Sindhi in Sindhi script)")
LAST = {"plan": ""}          # English plan, used to ground the advisor

# ---------- fixed UI labels ----------
UI = {
    "Urdu": {
        "title": "# 🌾 کسان سہولت: ایک کھیت، چار جانچ\nزمین • فصل • منافع • رہنمائی، آپ کی اپنی زبان میں",
        "district": "ضلع", "crop": "فصل", "acres": "ایکڑ",
        "issue": "رپورٹ شدہ مسئلہ (اختیاری)",
        "offer": "خریدار کی پیش کردہ قیمت فی من (0 = کوئی نہیں)",
        "plan_btn": "میرا فارم پلان بنائیں", "plan_out": "آپ کا فارم پلان",
        "img": "پتے کی تصویر اپلوڈ کریں", "photo_btn": "میری فصل چیک کریں", "photo_out": "نتیجہ",
        "q": "آپ کا سوال", "q_ph": "کھاد کدوں پاواں؟", "ask_btn": "پوچھیں", "ans_out": "جواب",
        "listen": "🔊 سنیں", "audio": "آڈیو",
        "lang": "زبان / Language",
    },
    "English": {
        "title": "# 🌾 Kisan Sahulat: one farm, four checks\nLand • Crop • Money • Guidance, in your language",
        "district": "District", "crop": "Crop", "acres": "Acres",
        "issue": "Reported issue (optional)",
        "offer": "Buyer's offered price per maund (0 = none)",
        "plan_btn": "Build my farm plan", "plan_out": "Your farm plan",
        "img": "Upload a leaf photo", "photo_btn": "Check my crop", "photo_out": "Result",
        "q": "Your question", "q_ph": "When should I apply fertilizer?", "ask_btn": "Ask", "ans_out": "Answer",
        "listen": "🔊 Listen", "audio": "Audio",
        "lang": "زبان / Language",
    },
}
def T(key, lang):
    # All non-English languages reuse the Urdu labels (same Arabic script)
    return UI["English" if lang == "English" else "Urdu"][key]

# ---------- voice (uses the voices from Cell 7) ----------
VOICE_MAP = {"Urdu": "ur-PK-UzmaNeural", "Punjabi": "ur-PK-UzmaNeural",
             "Sindhi": "ur-PK-UzmaNeural", "English": "en-US-AriaNeural"}
VOICE_MAP.update(voice_lib.REGIONAL_VOICES)          # Balochi, Saraiki, Brahui from Cell 7
VOICE_MAP["Pashto"] = "ps-AF-LatifaNeural"           # real Pashto voice; falls back to Urdu voice if missing
GTTS_MAP = {"Urdu": "ur", "English": "en", "Punjabi": "ur", "Sindhi": "ur"}
GTTS_MAP.update(voice_lib.GTTS_CODES)

def _ok(path):
    return os.path.exists(path) and os.path.getsize(path) > 500

def speak(text, lang):
    clean = re.sub(r"[*#_`>|]", "", str(text)).strip()[:1200]
    if not clean:
        return None
    fname = f"voice_{lang.lower()}_{hashlib.md5(clean.encode('utf-8')).hexdigest()[:8]}.mp3"
    if _ok(fname):
        return fname
    for v in dict.fromkeys([VOICE_MAP.get(lang, "ur-PK-UzmaNeural"), "ur-PK-UzmaNeural"]):
        try:
            voice_lib._run_async(voice_lib._edge_save(clean, v, fname))
            if _ok(fname):
                return fname
        except Exception:
            pass
    try:
        gTTS(text=clean, lang=GTTS_MAP.get(lang, "ur"), slow=False).save(fname)
        if _ok(fname):
            return fname
    except Exception:
        pass
    return None

def listen(text, lang):
    if not str(text).strip():
        gr.Warning("Nothing to read yet. Generate a result first.")
        return None
    path = speak(text, lang)
    if path is None:
        gr.Warning("Voice is unavailable right now (needs internet).")
    return path

# ---------- translation with disk cache (works offline once warmed) ----------
CACHE_FILE = "translation_cache.json"
_cache = {}
if os.path.exists(CACHE_FILE):
    try:
        _cache = json.load(open(CACHE_FILE, encoding="utf-8"))
    except Exception:
        _cache = {}

def _save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(_cache, f, ensure_ascii=False)

def _chunks(text, limit=1500):
    parts, cur = [], ""
    for para in text.split("\n"):
        if len(cur) + len(para) + 1 > limit and cur:
            parts.append(cur); cur = ""
        cur += para + "\n"
    if cur.strip():
        parts.append(cur)
    return parts

def translate(text, lang):
    if lang == "English" or not str(text).strip():
        return text
    key = hashlib.md5(f"{lang}|{text}".encode("utf-8")).hexdigest()
    if key in _cache:
        return _cache[key]
    system = (f"You translate farm reports for Pakistani farmers into {lang} {SCRIPT_NOTE}. "
              "Use simple, spoken, farmer-friendly words. Keep every number, unit (mm, %, PKR, "
              "maund, acre), district name and chemical active-ingredient name exactly right. "
              "Keep the line breaks and bullet structure. Do not add advice, brands or doses. "
              "Output ONLY the translation.")
    try:
        out = "\n".join(llm_answer(system, c, model=GEMINI_MODEL).strip() for c in _chunks(text))
        _cache[key] = out
        _save_cache()
        return out
    except Exception as e:
        return "ترجمہ دستیاب نہیں، انگریزی دکھائی جا رہی ہے۔\n\n" + text

# ---------- tab 1: farm plan ----------
def make_plan(district, crop, acres, issue, offer, lang):
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            ret = generate_farmer_action_plan(
                district, crop, float(acres),
                issue=None if issue in (None, "None") else issue,
                offered_price=float(offer) if offer and float(offer) > 0 else None)
    except Exception as e:
        return f"Could not build the plan: {e}"
    text = buf.getvalue().strip() or (ret if isinstance(ret, str) else str(ret))
    LAST["plan"] = text                      # keep English for grounding the advisor
    return translate(text, lang)

# ---------- tab 2: leaf photo (Gemini vision) ----------
def diagnose_photo(path, crop, lang):
    if not path:
        return "Please upload a leaf photo first." if lang == "English" else "پہلے پتے کی تصویر اپلوڈ کریں۔"
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    data = base64.b64encode(open(path, "rb").read()).decode()
    prompt = (f"This is a photo of a {crop} plant from Pakistan. Say what disease, pest or "
              "nutrient problem it most likely shows, how confident you are (low/medium/high), "
              "and 3 short next steps. Name active ingredients only, no brand names, no doses. "
              "End by advising a local agriculture officer to confirm. Under 120 words. "
              f"Write the whole answer in {lang} {SCRIPT_NOTE}, using simple farmer-friendly words.")
    body = {"contents": [{"role": "user", "parts": [
        {"text": prompt}, {"inline_data": {"mime_type": mime, "data": data}}]}],
        "generationConfig": {"maxOutputTokens": 800, "temperature": 0.2}}
    try:
        r = requests.post(f"{BASE}/models/{GEMINI_MODEL}:generateContent",
                          headers=HEAD, json=body, timeout=90)
        r.raise_for_status()
        return r.json()["candidates"][0]["content"]["parts"][0]["text"]
    except Exception as e:
        return f"Photo analysis unavailable right now ({str(e)[:80]}). Ask a local agriculture officer."

# ---------- tab 3: multilingual advisor ----------
def ask_advisor(question, lang):
    if not question.strip():
        return "Please type a question."
    if not LAST["plan"]:
        return "Please generate a farm plan first (tab 1), so I can use your farm's data."
    system = (f"You are Kisan Sahulat, a farming advisor for Pakistani farmers. Answer ONLY in {lang} "
              f"{SCRIPT_NOTE}. Use ONLY the farm data below. Never give pesticide doses or brand "
              "names. Stay on farming. If it is an emergency, tell them to contact a local "
              f"agriculture officer. Under 120 words.\n\nFARM DATA:\n{LAST['plan']}")
    try:
        return llm_answer(system, question, model=GEMINI_MODEL)
    except Exception as e:
        return f"Advisor unavailable right now ({str(e)[:80]})."

# ---------- UI ----------
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Nastaliq+Urdu&display=swap');
.urdu textarea, .urdu input, .urdu label span, .urdu button, .urdu p, .urdu h1 {
  font-family: 'Noto Nastaliq Urdu', 'Noto Naskh Arabic', sans-serif !important;
  unicode-bidi: plaintext; text-align: start; line-height: 2.1;
}
"""

with gr.Blocks(title="Kisan Sahulat", css=CSS) as demo:
    lang = gr.Radio(LANG_CHOICES, value="Urdu", label=T("lang", "Urdu"))
    title = gr.Markdown(T("title", "Urdu"), elem_classes="urdu")

    with gr.Tab("1. فارم پلان / Farm plan"):
        with gr.Row():
            d = gr.Dropdown(DISTRICT_LIST, value=DISTRICT_LIST[0], label=T("district", "Urdu"), elem_classes="urdu")
            c = gr.Dropdown(CROP_LIST, value=CROP_LIST[0], label=T("crop", "Urdu"), elem_classes="urdu")
            a = gr.Number(value=5, label=T("acres", "Urdu"), elem_classes="urdu")
        with gr.Row():
            i = gr.Dropdown(ISSUES, value="None", label=T("issue", "Urdu"), elem_classes="urdu")
            o = gr.Number(value=0, label=T("offer", "Urdu"), elem_classes="urdu")
        plan_btn = gr.Button(T("plan_btn", "Urdu"), variant="primary", elem_classes="urdu")
        plan_out = gr.Textbox(label=T("plan_out", "Urdu"), lines=26, elem_classes="urdu")
        plan_listen = gr.Button(T("listen", "Urdu"), elem_classes="urdu")
        plan_audio = gr.Audio(label=T("audio", "Urdu"), type="filepath", interactive=False)
        plan_btn.click(make_plan, [d, c, a, i, o, lang], plan_out)
        plan_listen.click(listen, [plan_out, lang], plan_audio)

    with gr.Tab("2. پتے کی تصویر / Leaf photo"):
        img = gr.Image(type="filepath", label=T("img", "Urdu"), elem_classes="urdu")
        pc = gr.Dropdown(CROP_LIST, value=CROP_LIST[0], label=T("crop", "Urdu"), elem_classes="urdu")
        photo_btn = gr.Button(T("photo_btn", "Urdu"), variant="primary", elem_classes="urdu")
        photo_out = gr.Textbox(label=T("photo_out", "Urdu"), lines=8, elem_classes="urdu")
        photo_listen = gr.Button(T("listen", "Urdu"), elem_classes="urdu")
        photo_audio = gr.Audio(label=T("audio", "Urdu"), type="filepath", interactive=False)
        photo_btn.click(diagnose_photo, [img, pc, lang], photo_out)
        photo_listen.click(listen, [photo_out, lang], photo_audio)

    with gr.Tab("3. سوال پوچھیں / Ask"):
        q = gr.Textbox(label=T("q", "Urdu"), placeholder=T("q_ph", "Urdu"), elem_classes="urdu")
        ask_btn = gr.Button(T("ask_btn", "Urdu"), variant="primary", elem_classes="urdu")
        ans_out = gr.Textbox(label=T("ans_out", "Urdu"), lines=8, elem_classes="urdu")
        ans_listen = gr.Button(T("listen", "Urdu"), elem_classes="urdu")
        ans_audio = gr.Audio(label=T("audio", "Urdu"), type="filepath", interactive=False)
        ask_btn.click(ask_advisor, [q, lang], ans_out)
        ans_listen.click(listen, [ans_out, lang], ans_audio)

    # switching language relabels every field instantly (no API call)
    def on_lang(l):
        return [
            gr.update(value=T("title", l)),
            gr.update(label=T("district", l)), gr.update(label=T("crop", l)),
            gr.update(label=T("acres", l)), gr.update(label=T("issue", l)),
            gr.update(label=T("offer", l)),
            gr.update(value=T("plan_btn", l)), gr.update(label=T("plan_out", l)),
            gr.update(value=T("listen", l)), gr.update(label=T("audio", l)),
            gr.update(label=T("img", l)), gr.update(label=T("crop", l)),
            gr.update(value=T("photo_btn", l)), gr.update(label=T("photo_out", l)),
            gr.update(value=T("listen", l)), gr.update(label=T("audio", l)),
            gr.update(label=T("q", l), placeholder=T("q_ph", l)),
            gr.update(value=T("ask_btn", l)), gr.update(label=T("ans_out", l)),
            gr.update(value=T("listen", l)), gr.update(label=T("audio", l)),
        ]
    lang.change(on_lang, lang,
                [title, d, c, a, i, o, plan_btn, plan_out, plan_listen, plan_audio,
                 img, pc, photo_btn, photo_out, photo_listen, photo_audio,
                 q, ask_btn, ans_out, ans_listen, ans_audio])

demo.launch(share=True, debug=False)

/tmp/ipykernel_459/599176867.py:210: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Kisan Sahulat", css=CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4f882605bd326a00b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
